In [1]:
import torch
from torchvision.datasets import CIFAR10
from torchvision import transforms, models
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Subset, ConcatDataset

In [2]:
from torch.utils.data import Dataset
class AdvDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __getitem__(self, index):
        return self.x[index].permute(2,0,1).cpu(), self.y[index].item()

    def __len__(self):
        return len(self.x)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
transform_cifar = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.ToTensor()
])

In [5]:
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
train_dataset = Subset(train_dataset, range(30000))
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform_test)
# test_dataset = Subset(test_dataset, range(5000)).dataset

# print(f'CIFAR10: train {train_dataset.data.shape}, test {test_dataset.data.shape}')

train_dataset_adv = torch.load('./adv_data/resnet_pgd_cifar_train')
train_dataset_adv = Subset(train_dataset_adv, range(30000))
# test_dataset_adv = torch.load('./adv_data/resnet_pgd_cifar_test')
# test_dataset_adv = Subset(test_dataset_adv, range(5000)).dataset
# 
train_dataset = ConcatDataset([train_dataset, train_dataset_adv])
# test_dataset = ConcatDataset([test_dataset, test_dataset_adv])


Files already downloaded and verified
Files already downloaded and verified


In [6]:
batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [7]:
print(f'CIFAR10: train {len(train_loader)}, test {len(test_loader)}')

CIFAR10: train 469, test 79


In [8]:
model = models.resnet50(weights="DEFAULT")

In [9]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [10]:
# modify the input and output layers
# model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
model.fc = nn.Linear(2048, 10)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [11]:
lr = 0.01

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
best_val_loss = float('inf')
epochs = 10

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    tarin_bar = tqdm(train_loader, position=0, leave=True)
    for x, y in tarin_bar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        train_correct += class_pred.eq(y).sum().item()
    train_accuracy = train_correct / len(train_dataset)

    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_bar = tqdm(test_loader, position=0, leave=True)
    with torch.no_grad():
        for x, y in val_bar:
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = criterion(y_pred, y)
            val_loss += loss.item()
            class_pred = y_pred.argmax(dim=1)
            val_correct += class_pred.eq(y).sum().item()
    val_accuracy = val_correct / len(test_dataset)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), './model/ResNet50_CIFAR_pgd.pth')
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.6f}, Train Acc: {train_accuracy:.6f}, Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')
  

100%|██████████| 79/79 [00:01<00:00, 52.01it/s]


Epoch 1/10, Train Loss: 1.193084, Train Acc: 0.576717, Val Loss: 0.721967, Val Acc: 0.749500


100%|██████████| 79/79 [00:01<00:00, 56.16it/s]


Epoch 2/10, Train Loss: 0.712508, Train Acc: 0.749633, Val Loss: 0.610059, Val Acc: 0.788000


100%|██████████| 79/79 [00:01<00:00, 52.41it/s]


...

Epoch 8/10, Train Loss: 0.183581, Train Acc: 0.937617, Val Loss: 0.641919, Val Acc: 0.823500


100%|██████████| 79/79 [00:01<00:00, 55.02it/s]


Epoch 9/10, Train Loss: 0.155400, Train Acc: 0.947117, Val Loss: 0.647673, Val Acc: 0.828600


100%|██████████| 79/79 [00:01<00:00, 53.82it/s]

Epoch 10/10, Train Loss: 0.135922, Train Acc: 0.954767, Val Loss: 0.661980, Val Acc: 0.826100


In [12]:
# model = models.resnet50()
# # modify the input and output layers
# # model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
# model.fc = nn.Linear(2048, 10)
model.load_state_dict(torch.load('model/ResNet50_CIFAR_pgd.pth'))
model = model.to(device)

In [13]:
model.eval()
val_loss = 0
val_correct = 0
val_bar = tqdm(test_loader, position=0, leave=True)
with torch.no_grad():
    for x, y in val_bar:
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = criterion(y_pred, y)
        val_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        val_correct += class_pred.eq(y).sum().item()
val_accuracy = val_correct / len(test_dataset)

print(f'Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')

100%|██████████| 79/79 [00:01<00:00, 54.83it/s]

Val Loss: 0.565187, Val Acc: 0.807000


In [14]:

test_dataset_adv = torch.load('./adv_data/resnet_pgd_cifar_test')

test_loader_adv = DataLoader(test_dataset_adv, batch_size=batch_size, shuffle=False)

In [15]:
model.eval()
val_loss = 0
val_correct = 0
val_bar = tqdm(test_loader_adv, position=0, leave=True)
with torch.no_grad():
    for x, y in val_bar:
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = criterion(y_pred, y)
        val_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        val_correct += class_pred.eq(y).sum().item()
val_accuracy = val_correct / len(test_dataset)

print(f'Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')

100%|██████████| 79/79 [00:01<00:00, 62.02it/s]

Val Loss: 0.912826, Val Acc: 0.694100
